Modeling

Suggested I could have up to 100 models.

I will make several variants of each model type. They will vary by GridSearch optimization, feature selection, training slice, and more.

Each model will be saved as a joblib file in a dedicated folder to be loaded later.

The joblib files will be named with parameters, and a table/sheet/dictionary will also be created with more detailed explanations of each.

Linear Regression,Random Forest Regression, SVM, XGBoost, LSTM, Prophet were specifically named in the instruction. Should also do more.

What is my target? Should do regression, or merely classification?

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import median_absolute_error
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score, KFold

# This notebook will concern modeling raw daily data with no additional calculated metrics. Additional notebooks will specify the conditions for creating more metrics, choosing sets of features, and modeling.

In [15]:
df_daily_data = pd.read_csv('transformed_daily_data/daily_data_5y_30k_features1.csv')
df_daily_data

,timestamp,ticker,open,high,low,close,volume,vwap,transactions,close_vs_open_pct,...,SMA50,SMA100,RSI,MACD,MACD_signal,MACD_hist,rolling_volatility,close_lag1,close_lag2,return_lag1
0,2021-07-15,AA,35.27,35.9300,34.2350,34.54,8.679014e+06,34.9029,59127.0,-0.020697,...,37.3496,37.3496,42.191544,-0.409851,-0.364569,-0.045282,0.038311,35.14,35.62,-0.013476
1,2021-07-16,AA,35.35,35.7000,32.7323,32.95,1.478064e+07,33.4495,91473.0,-0.067893,...,37.1966,37.1966,38.717340,-0.620150,-0.415686,-0.204464,0.019171,34.54,35.14,-0.017075
2,2021-07-19,AA,31.99,32.4500,30.9950,31.62,1.240088e+07,31.6877,83015.0,-0.011566,...,37.0312,37.0312,33.650329,-0.883943,-0.509337,-0.374606,0.015777,32.95,34.54,-0.046034
3,2021-07-20,AA,31.55,35.2602,31.2600,34.95,1.578655e+07,34.1895,99391.0,0.107765,...,36.9062,36.9062,41.709512,-0.814904,-0.570450,-0.244454,0.061817,31.62,32.95,-0.040364
4,2021-07-21,AA,35.25,37.4100,35.2500,37.09,1.200015e+07,36.7297,74959.0,0.052199,...,36.8272,36.8272,50.734862,-0.580815,-0.572523,-0.008292,0.067239,34.95,31.62,0.105313
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
478380,2026-04-27,ZTS,116.62,119.6800,116.6000,117.87,3.160355e+06,118.2275,43149.0,0.010719,...,120.8320,120.8320,52.347521,-0.496987,-0.376874,-0.120113,0.018470,116.87,116.06,0.006979
478381,2026-04-28,ZTS,117.23,118.2900,116.0750,116.65,2.972547e+06,117.0538,47388.0,-0.004948,...,120.6320,120.6320,42.640076,-0.606465,-0.422792,-0.183673,0.009769,117.87,116.87,0.008557
478382,2026-04-29,ZTS,115.50,115.9050,112.9700,113.83,3.401808e+06,113.9745,47839.0,-0.014459,...,120.3974,120.3974,36.971235,-0.910285,-0.520291,-0.389995,0.013879,116.65,117.87,-0.010350
478383,2026-04-30,ZTS,114.04,115.5350,113.7900,114.97,4.297206e+06,114.7143,57409.0,0.008155,...,120.1142,120.1142,43.620309,-1.047007,-0.625634,-0.421373,0.014981,113.83,116.65,-0.024175


# Random Forest


In [3]:
%%time

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

df_model = df_daily_data.copy()

# Create target correctly by ticker
next_close = df_model.groupby('ticker')['close'].shift(-1)
df_model['target'] = (next_close > df_model['close']).astype(int)

# Remove final row per ticker where next_close does not exist
df_model = df_model[next_close.notna()].copy()

# Columns to exclude from features
exclude_cols = ['timestamp', 'ticker', 'target']

feature_cols = [col for col in df_model.columns if col not in exclude_cols]

# Features and target
X = df_model[feature_cols]
y = df_model['target']

# Basic train/test split for now
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42
)

model_rfc_daily_GSX_featmax = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

model_rfc_daily_GSX_featmax.fit(X_train, y_train)

y_pred = model_rfc_daily_GSX_featmax.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.5110253352580598
Precision: 0.518363783750078
Recall: 0.5140789907651715
F1: 0.5162124959895678
Confusion Matrix:
[[23914 23172]
 [23573 24939]]
CPU times: user 8min 41s, sys: 4.01 s, total: 8min 45s
Wall time: 1min 16s


# Testing on 2025 and 2026

In [3]:
%%time

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

df_model = df_daily_data.copy()

# Ensure datetime
df_model['timestamp'] = pd.to_datetime(df_model['timestamp'])

# =========================
# TARGET
# =========================
next_close = df_model.groupby('ticker')['close'].shift(-1)

df_model['target'] = (
    next_close > df_model['close']
).astype(int)

# Remove last row per ticker
df_model = df_model[next_close.notna()].copy()

# =========================
# FEATURES
# =========================
exclude_cols = ['timestamp', 'ticker', 'target']

feature_cols = [
    col for col in df_model.columns
    if col not in exclude_cols
]

X = df_model[feature_cols]
y = df_model['target']

# =========================
# TIME-BASED SPLIT
# Train: 2021-2024
# Test: 2025
# =========================
train_mask = df_model['timestamp'] < '2025-01-01'
test_mask = df_model['timestamp'] >= '2025-01-01'

X_train = X[train_mask]
X_test = X[test_mask]

y_train = y[train_mask]
y_test = y[test_mask]

# =========================
# MODEL
# =========================
model_rfc_daily_GSX_featmax = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

model_rfc_daily_GSX_featmax.fit(X_train, y_train)

# =========================
# PREDICTIONS
# =========================
y_pred = model_rfc_daily_GSX_featmax.predict(X_test)

# =========================
# EVALUATION
# =========================
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.5040894054808656
Precision: 0.5174936436381787
Recall: 0.4907256752359258
F1: 0.5037543180351517

Confusion Matrix:
[[33265 30933]
 [34430 33176]]
CPU times: user 6min 18s, sys: 3.04 s, total: 6min 21s
Wall time: 55.6 s


# Train 2025, hold out 2026 for validation

In [16]:
%%time

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

df_model = df_daily_data.copy()

# Ensure datetime
df_model['timestamp'] = pd.to_datetime(df_model['timestamp'])

# =========================
# TARGET
# =========================
next_close = df_model.groupby('ticker')['close'].shift(-1)

df_model['target'] = (
    next_close > df_model['close']
).astype(int)

# Remove last row per ticker
df_model = df_model[next_close.notna()].copy()

# =========================
# FEATURES
# =========================
exclude_cols = ['timestamp', 'ticker', 'target']

feature_cols = [
    col for col in df_model.columns
    if col not in exclude_cols
]

X = df_model[feature_cols]
y = df_model['target']


# Time-based split
train_mask = df_model['timestamp'].dt.year.between(2021, 2024)
test_mask = df_model['timestamp'].dt.year == 2025
val_mask = df_model['timestamp'].dt.year == 2026

X_train = X[train_mask]
y_train = y[train_mask]

X_test = X[test_mask]
y_test = y[test_mask]

X_val = X[val_mask]
y_val = y[val_mask]

# =========================
# MODEL
# =========================
model_rfc_daily_GSX_featmax = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1
)

model_rfc_daily_GSX_featmax.fit(X_train, y_train)

y_test_pred = model_rfc_daily_GSX_featmax.predict(X_test)
y_val_pred = model_rfc_daily_GSX_featmax.predict(X_val)

# =========================
# PREDICTIONS
# =========================
y_pred = model_rfc_daily_GSX_featmax.predict(X_test)

# =========================
# EVALUATION
# =========================
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.5126448362720403
Precision: 0.5187962407518496
Recall: 0.7121877573428493
F1: 0.6003007866728366

Confusion Matrix:
[[14557 33691]
 [14679 36323]]
CPU times: user 4min 4s, sys: 1.83 s, total: 4min 6s
Wall time: 33.9 s


In [5]:
y_pred

array([1, 1, 0, ..., 1, 0, 1])

# 2025 Results dataframe

In [17]:
# =========================
# BUILD 2025 RESULTS DF
# =========================

# Get 2025 rows
df_2025_results = df_model.loc[test_mask].copy()

# Add predictions
df_2025_results['prediction'] = y_pred

# Optional: prediction probabilities
df_2025_results['prediction_probability_up'] = (
    model_rfc_daily_GSX_featmax.predict_proba(X_test)[:, 1]
)

# Optional: correctness flag
df_2025_results['prediction_correct'] = (
    df_2025_results['prediction'] == df_2025_results['target']
).astype(int)

# Optional: readable labels
df_2025_results['prediction_label'] = df_2025_results['prediction'].map({
    1: 'UP',
    0: 'DOWN'
})

df_2025_results['actual_label'] = df_2025_results['target'].map({
    1: 'UP',
    0: 'DOWN'
})

# Reset index for cleaner viewing
df_2025_results = df_2025_results.reset_index(drop=True)

df_2025_results

,timestamp,ticker,open,high,low,close,volume,vwap,transactions,close_vs_open_pct,...,rolling_volatility,close_lag1,close_lag2,return_lag1,target,prediction,prediction_probability_up,prediction_correct,prediction_label,actual_label
0,2025-01-02,AA,38.165,39.0400,37.9000,37.99,2703886.0,38.3283,33119.0,-0.004585,...,0.014294,37.78,37.15,0.016958,0,1,0.510494,0,UP,DOWN
1,2025-01-03,AA,37.950,37.9500,35.3752,35.71,7491799.0,35.9337,63814.0,-0.059025,...,0.029454,37.99,37.78,0.005558,1,0,0.490390,0,DOWN,UP
2,2025-01-06,AA,36.000,37.0750,35.9200,36.49,5983917.0,36.4794,58230.0,0.013611,...,0.033236,35.71,37.99,-0.060016,0,1,0.508881,0,UP,DOWN
3,2025-01-07,AA,36.890,37.3000,35.7550,36.24,3381466.0,36.2093,39434.0,-0.017620,...,0.032951,36.49,35.71,0.021843,0,1,0.510468,0,UP,DOWN
4,2025-01-08,AA,35.780,36.0300,34.7500,36.00,3775303.0,35.5672,41012.0,0.006149,...,0.030724,36.24,36.49,-0.006851,0,1,0.506493,0,UP,DOWN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99245,2025-12-24,ZTS,123.100,125.6899,123.0600,125.49,2369020.0,125.1484,29115.0,0.019415,...,0.008772,123.54,123.78,-0.001939,1,1,0.513344,1,UP,UP
99246,2025-12-26,ZTS,125.160,126.3200,124.8000,126.23,3226934.0,125.8979,45295.0,0.008549,...,0.008752,125.49,123.54,0.015784,0,1,0.515024,0,UP,DOWN
99247,2025-12-29,ZTS,126.160,126.8500,125.5607,125.98,4465924.0,126.0785,51439.0,-0.001427,...,0.008154,126.23,125.49,0.005897,1,1,0.514729,1,UP,UP
99248,2025-12-30,ZTS,125.550,127.6000,125.4500,126.41,3230541.0,126.5667,45221.0,0.006850,...,0.007307,125.98,126.23,-0.001981,0,1,0.508025,0,UP,DOWN


# Feature correlation to correctness

In [18]:
# =========================
# ERROR FLAG
# 1 = incorrect prediction
# 0 = correct prediction
# =========================

df_2025_results['incorrect_prediction'] = (
    df_2025_results['prediction'] != df_2025_results['target']
).astype(int)

# =========================
# NUMERIC FEATURES ONLY
# =========================

numeric_cols = df_2025_results.select_dtypes(include=['number']).columns

# Remove columns you don't want in correlation analysis
exclude_corr_cols = [
    'target',
    'prediction',
    'prediction_correct',
    'incorrect_prediction'
]

corr_feature_cols = [
    col for col in numeric_cols
    if col not in exclude_corr_cols
]

# =========================
# CORRELATION TO INCORRECT PREDICTIONS
# =========================

feature_error_corr = (
    df_2025_results[corr_feature_cols]
    .corrwith(df_2025_results['incorrect_prediction'])
    .sort_values(ascending=False)
    .to_frame(name='corr_to_incorrect_prediction')
)

feature_error_corr.sort_values(by='corr_to_incorrect_prediction')

,corr_to_incorrect_prediction
prediction_probability_up,-0.004590
volume,-0.002902
transactions,0.000343
SMA50,0.000661
SMA100,0.000661
SMA10,0.000852
SMA20,0.000900
close_lag2,0.000941
low,0.001043
open,0.001192


In [19]:
feature_error_corr

,corr_to_incorrect_prediction
daily_high_vs_low_pct,0.018752
rolling_volatility,0.012105
prev_day_return,0.009904
close_vs_open_pct,0.009379
return_lag1,0.005842
MACD,0.004262
RSI,0.004203
MACD_signal,0.003553
MACD_hist,0.003029
high,0.001342


In [20]:
import joblib

joblib.dump(model_rfc_daily_GSX_featmax, 'models_daily/model_daily_rfc_GSX_featmax.joblib')


['models_daily/model_daily_rfc_GSX_featmax.joblib']

# A previous run resulted in a 1GB file dump. That seemed wrong.

In [10]:
model_rfc_daily_GSX_featmax.n_estimators


100

In [12]:
model_rfc_daily_GSX_featmax.max_depth

In [13]:
sum(estimator.tree_.node_count for estimator in model_rfc_daily_GSX_featmax.estimators_)

13065226

In [ ]:
sum(estimator.tree_.node_count for estimator in rf_model.estimators_)